# Training Series: Introduction to Discontinuous Galerkin Methods for Flow Problems <br> - Implementation of a DG solver for incompressible single-phase flows

In [1]:
//#r "../BoSSSpad/BoSSSpad.dll"
#r "C:\Program Files (x86)\FDY\BoSSS\bin\Release\net6.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.IO;
using System.Linq;
using System.Runtime.Serialization;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

## Problem statement: transient incompressible single-phase flow

The flow within the computational domain $\Omega \in R^2$ is described by the unsteady Navier-Stokes equations 

$$ \rho_f\left(\frac{\partial \vec{u}}{\partial t}+ \vec{u} \cdot \nabla \vec{u}\right) = -\nabla p + \mu_f \Delta \vec{u} = \rho_f \vec{f} \quad \textrm{in}\ \Omega $$

$$ \rho_f\left(\frac{\partial \vec{u}}{\partial t}+ \nabla \cdot \left( \vec{u} \otimes \vec{u} \right) \right) 
= \nabla \cdot \left( -p\mathbf{I} + \mu_f \left( \nabla \vec{u} + \nabla \vec{u}^{\textrm{T}} \right) \right) = \rho_f \vec{f} \quad \textrm{in}\ \Omega $$

and the continuity equation
$$ \nabla \cdot \vec{u} = 0 \quad \forall\ t \in (0,T)\quad \textrm{in}\ \Omega $$

In the equations above 
 - $\vec{u}$ is the velocity vector 
 - $p$ the pressure
 - and $\vec{f}$ some volume force, e.g. gravity. 
 - The fluid density is denoted by $\rho_f$
 - $\mu_f=\rho_f \cdot \nu_f$ is the dynamic viscosity of the fluid.
 
At the boundary $\partial \Omega = \partial \Omega_\textrm{D} \cup \partial \Omega_\textrm{N}$ the corresponding boundary conditions read 

$$ \vec{u} = \vec{u}_\textrm{D} \quad \textrm{on}\ \partial \Omega_\textrm{D} $$

$$ -p \vec{n}_{\partial \Omega} + \mu_f \left( \nabla \vec{u} + \nabla \vec{u}^{\textrm{T}} \right) \vec{n}_{\partial \Omega} = - p_\textrm{ext} \vec{n}_{\partial \Omega} \quad \textrm{on}\ \partial \Omega_\textrm{N} $$

## Setting up test cases

In [2]:
using System.IO;
using BoSSS.Solution.NSECommon;

In [3]:
// remove old plot files
public static void DeletePlotFilesInCurrentDirectory() {
    var dir = new DirectoryInfo(Directory.GetCurrentDirectory());
    Console.Write("rm");
    foreach (var pltFile in dir.GetFiles("XNSE-*")) {
        Console.Write(" " + pltFile.Name);
        pltFile.Delete();
    }
    foreach (var resFile in dir.GetFiles("residual-L2.*")) {
        Console.Write(" " + resFile.Name);
        resFile.Delete();
    }
    Console.WriteLine(";");
}

## Test case 1 - Channel flow

In [4]:
XNSE_Control C1 = new XNSE_Control();

// database and saving options
C1.DbPath = null;
C1.savetodb = false;
C1.ProjectName = "ChannelFlow";
C1.ImmediatePlotPeriod = 1;
C1.SuperSampling = 3;

// Physical parameters
C1.PhysicalParameters.rho_A = 1.0;
C1.PhysicalParameters.mu_A = 1.0 / 10.0;
C1.PhysicalParameters.IncludeConvection = false;    // false -> Stokes

// DG degree
C1.SetDGdegree(3);

// computational grid
C1.GridFunc = delegate {
    int gridResolution = 2;

    double Length = 10.0;
    double Height = 2.0;

    double[] _xNodes = GenericBlas.Linspace(0, Length, gridResolution * 5 + 1);
    double[] _yNodes = GenericBlas.Linspace(-Height/2.0, Height/2.0, gridResolution + 1);

    var grd = Grid2D.Cartesian2DGrid(_xNodes, _yNodes, BoSSS.Foundation.Grid.RefElements.CellType.Square_Linear);

    grd.DefineEdgeTags(delegate (double[] _X) {
        double x = _X[0];
        double y = _X[1];

    if(Math.Abs(y - (-Height/2.0)) < 1.0e-6)
        // bottom
        return IncompressibleBcType.Wall.ToString() + "_bottom";

    if(Math.Abs(y - (+Height/2.0)) < 1.0e-6)
        // top
        return IncompressibleBcType.Wall.ToString()+ "_top";


    if(Math.Abs(x - (0.0)) < 1.0e-6)
        // left
        return IncompressibleBcType.Velocity_Inlet.ToString();

    if(Math.Abs(x - (Length)) < 1.0e-6)
        // right
        return IncompressibleBcType.Pressure_Outlet.ToString();

        throw new ArgumentOutOfRangeException();
    });

    return grd;
};

// set boundary conditions
C1.AddBoundaryValue(IncompressibleBcType.Velocity_Inlet.ToString(), "VelocityX", new Formula("X => 1 - X[1] * X[1]", TimeDep: false));

// Set Initial Conditions
C1.AddInitialValue("VelocityX", new Formula("X => 0.0", TimeDep: false));

// Solver Options
bool transient = true;
if(transient) {
    C1.NoOfTimesteps = 20;
    C1.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    C1.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.ImplicitEuler;
    C1.dtFixed = 0.1;
} else {
    C1.TimesteppingMode = AppControl._TimesteppingMode.Steady;
    C1.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.ImplicitEuler;
}


Note: Since the solver may be executed in an external program, the control object has to be saved in a file. 
For lots of complicated objects, especially for delegates, C\# does not support serialization (converting the object into a form that can be saved on disk, or transmitted over a network), so a workaround is needed.
This is achieved e.g. by the **Formula** object, where a C\#-formula is saved as a string.

In [5]:
// remove old plot files
DeletePlotFilesInCurrentDirectory();

rm residual-L2.2024Oct16_14-18-41.txt residual-L2.2024Oct16_15-08-48.txt residual-L2.2024Oct21_16-22-40.txt;


In [6]:
C1.Run()

Grid Edge Tags changed.
Session ID: 00000000-0000-0000-0000-000000000000, DB path: 'EMPTY'
Session directory 'EMPTY\sessions\00000000-0000-0000-0000-000000000000'.
Grid repartitioning method: METIS
Grid repartitioning options: 
Number of cell Weights: 0
Going with agglomeration threshold: 0.1
Linearization hint: AdHoc
=============== Operator Configuration ===============
     isGravity                     :[ ]
     isVolForce                    :[ ]
     isTransport                   :[ ]
     isViscous                     :[x]
     isPressureGradient            :[x]
     isInterfaceSlip               :[ ]
     isContinuity                  :[x]
     isMovingMesh                  :[ ]
     isMatInt                      :[x]
     isPInterfaceSet               :[ ]
     isImmersedBoundary            :[ ]
     withPressureDissipation       :[ ]
=============== Linear Solver Configuration ===============
     Solvercode                    :Sparse direct solver PARDISO
=============== Nonl

NO-PROJ	NO-NAME-SET	10/21/2025 3:09:46 PM	00000000...

# Workflow Management and Meta Job Scheduler (BONUS)

This section illustrates how computations can be executed on large 
(or small) compute servers, aka.
high-performance-computers (HPC), aka. compute clusters, aka. supercomputers, etc.

This means, the computation is not executed on the local workstation 
(or laptop) but on some other computer.
This approach is particulary handy for large computations, which run
for multiple hours or days, since a user can 
e.g. shutdown or restart his personal computer without killing the compute job. 

*BoSSS* features a set of classes and routines 
(an API, application programing interface) for communication with 
compute clusters. This is especially handy for **scripting**,
e.g. for parameter studies, where dozens of computations 
have to be started and monitored.

### Batch Processing

First, we have to select a *batch system* (aka.*execution queue*, aka. *queue*) that we want to use.
Batch systems are a common approach to organize workloads (aka. compute jobs)
on compute clusters.
On such systems, a user typically does **not** starts a simulation manually/interactively.
Instead, he specifies a so-called *compute job*. The *scheduler* 
(i.e. the batch system) collects 
compute jobs from all users on the compute cluster, sorts them according to 
some priority and puts the jobs into some queue, also called *batch*.
The jobs in the batch are then executed in order, depending on the 
available hardware and the scheduling policies of the system.

The *BoSSS* API provides front-ends (clients) for the following 
batch system software:

- `BoSSS.Application.BoSSSpad.SlurmClient` for the 
Slurm Workload Manager (very prominent on Linux HPC systems)
- `BoSSS.Application.BoSSSpad.MsHPC2012Client`
for the Microsoft HPC Pack 2012 and higher
- `BoSSS.Application.BoSSSpad.MiniBatchProcessorClient` for the 
mini batch processor, a minimalistic, *BoSSS*-internal batch system which mimics 
a supercomputer batch system on the local machine.

A list of clients for various batch systems, which are loaded at the 
`Init()` command can be configured through the  
`~/.BoSSS/etc/BatchProcessorConfig.json`-file.
If this file is missing, a default setting, containing a 
mini batch processor, is initialized. 

The list of all execution queues can be accessed through:

In [7]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client LocalPC @C:\BoSSS-localJobs\binaries
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries"


### Note on the Mini Batch Processor:
The batch processor for local jobs can be started separately (by launching
`MiniBatchProcessor.exe` or `dotnet MiniBatchProcessor.dll`), **which is the preferred option**.
Alternatively, it can be started from Jupyter Notebook; it depends on the operating system, whether the 
`MiniBatchProcessor.exe` is terminated with the notebook kernel, or not.
If no mini-batch-processor is running, it is started (hopefully) upon Job activation.

## Initializing the workflow management

In order to use the workflow management, 
the very first thing we have to do is to initialize it by defineing 
a **project name**, here it is `IntroDG`. 
This is used to generate names for the compute jobs and to 
identify sessions in the database:

In [8]:
BoSSSshell.WorkflowMgm.Init("IntroDG");

Project name is set to 'IntroDG'.
Default Execution queue is chosen for the database.
Opening existing database '\\dc3\userspace\smuda\hpccluster\IntroDG'.


For this project, the default execution queue is set to:

In [9]:
GetDefaultQueue()

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
NumOfAdditionalServiceCores,0
NumOfAdditionalServiceCoresMPISerial,0
NumOfServiceCoresPerMPIprocess,0
ServerName,DC3


## Activation and Monitoring of the the Job

Finally, we are ready to deploy the job at the batch processor;
In a usual work flow scenario, we **do not** want to (re-) submit the 
job every time we run the worksheet -- usually, one wants to run a job once.

The concept to overcome this problem is job activation. If a job is 
activated, the meta job manager first checks the databases and the batch 
system, if a job with the respective name and project name is already 
submitted. Only if there is no information that the job was ever submitted
or started anywhere, the job is submitted to the respective batch system.

In [ ]:
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();  // check correlation by name
//BoSSSshell.WorkflowMgm.SetEqualityBasedSessionJobControlCorrelation();

First, a `Job` -object is created from the control object:

In [ ]:
C1.savetodb = true;

In [10]:
var JobLocal = C1.CreateJob();

This job is not activated yet, it can still be configured:

In [11]:
JobLocal.Status

PreActivation

### Starting the compute Job

One can change e.g. the number of MPI processes:

In [ ]:
JobLocal.NumberOfMPIProcs = 2;

Note that these jobs are desigend to be **persistent**:
This means the computation is only started 
**once for a given control object**, no matter how often the worksheet
is executed. 

Such a behaviour is useful for expensive simulations, which run on HPC
servers over days or even weeks. The user (you) can close the worksheet
and maybe open and execute it a few days later, and he can access
the original job which he submitted a few days ago (maybe it is finished
now).

Then, the job is activated, resp. submitted, resp. deployed 
to one batch system.
If job persistency is not wanted, traces of the job can be removed 
on request during activation, causing a fresh job deployment at the
batch system:

In [ ]:
JobLocal.Activate();  // execute the job in the default execution queue
//JobLocal.Activate(ExecutionQueues[4]);  // execute the job e.g. in queue 4

Deployments so far (0): ;
Success: 0
job submit count: 0
unable to determine job status - unknown
Deploying job SteadyStateChannel ... 
Opening existing database '\\dc3\userspace\smuda\hpccluster\IntroDG'.
Set Database: { Session Count = 1; Grid Count = 1; Path = D:\local\IntroDG }
Grid is not in database yet...
Grid successfully saved: ea53d8d8-191a-4c31-bf59-4e2549075f88
Deploying executables and additional files ...
Deployment directory: D:\local\binaries\IntroDG-XNSE_Solver2024Oct17_103002.828085
copied 42 files.
   written file: control.obj
deployment finished.
Starting mini batch processor in external process...
Started mini batch processor on local machine, process id is 30172.
started.


All jobs can be listed using the workflow management:

In [ ]:
BoSSSshell.WorkflowMgm.AllJobs

#0: SteadyStateChannel: FinishedSuccessful (MiniBatchProcessor client  LocalPC @D:\local\binaries)	SteadyStateChannel: FinishedSuccessful (MiniBatchProcessor client  LocalPC @D:\local\binaries)


Check the present job status:

In [ ]:
JobLocal.Status

FinishedSuccessful

### Evaluation of Job

We examine the output and error stream of the job:
This directly accesses the *\tt stdout*-redirection of the respective job
manager, which may contain a bit more information than the 
`Stdout`-copy in the session directory.

In [ ]:
JobLocal.Stdout

Session ID: 1ad77573-f453-4f66-9dac-f747ca732a02, DB path: 'D:\local\IntroDG'
Session directory 'D:\local\IntroDG\sessions\1ad77573-f453-4f66-9dac-f747ca732a02'.
Grid repartitioning method: METIS
Grid repartitioning options: 
Number of cell Weights: 0
noOfPartitioningsToChooseFrom = 1
Cells per rank: [160, 160]
Going with agglomeration threshold: 0.1
Linearization hint: AdHoc
=============== Operator Configuration ===============
     isGravity                     :[ ]
     isVolForce                    :[ ]
     isTransport                   :[ ]
     isViscous                     :[x]
     isPressureGradient            :[x]
     isInterfaceSlip               :[ ]
     isContinuity                  :[x]
     isMovingMesh                  :[ ]
     isMatInt                      :[x]
     isPInterfaceSet               :[ ]
     isImmersedBoundary            :[ ]
     withPressureDissipation       :[ ]
=============== Linear Solver Configuration ===============
     Solvercode           

Additionally we display the error stream and hope that it is empty:

In [ ]:
JobLocal.Stderr

We can also obtain the session 
which was stored during the execution of the job:

In [ ]:
var Sloc = JobLocal.LatestSession;
Sloc

IntroDG	SteadyStateChannel	10/17/2024 10:30:29	1ad77573...

## Exporting Plots

Each run of the solver corresponds to one **session** in the database. A session is basically a collection of information on the entire solver run, i.e. the simulation result, input and solver settings as well as meta-data such as computer and data and time.

Since in this tutorial only one solver run was executed, there is only one session in the Workflow Management (`wmg` is just an alias for `BoSSSshell.WorkflowMgm.`):

In [ ]:
wmg.Sessions

#0: IntroDG	SteadyStateChannel	10/17/2024 10:30:29	1ad77573...
#1: IntroDG	SteadyStateChannel	10/16/2024 20:07:16	19548988...


We select the first (and only) session and create an **export instruction** object.
The **supersampling** setting increases the output resolution. 

In [ ]:
var outPath = wmg.Sessions[0].Export().WithSupersampling(2).Do();

Starting export process... Data will be written to the directory: C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\IntroDG__SteadyStateChannel__1ad77573-f453-4f66-9dac-f747ca732a02


On the respective directory (see output above) one should finally find plot-files which than can be used
for further post processing in third-party software such as Paraview, LLNL Visit or Tecplot.

The `Do()` command returns the location of the output files:

In [ ]:
outPath

C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\IntroDG__SteadyStateChannel__1ad77573-f453-4f66-9dac-f747ca732a02